### Imports

In [3]:
import os
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, ConfusionMatrixDisplay

import torch
import torch.nn as nn
from torch.utils.data import DataLoader

from tqdm import tqdm

from data.kaggle_text import load_kaggle_disaster_csv
from models.text_branch import create_text_classifier, TextDatasetWrapper, TextClassificationWrapper
from training.utils import train_one_epoch, evaluate

ModuleNotFoundError: No module named 'data'

### Load and tokenize Kaggle dataset

In [ ]:
# Load Kaggle disaster tweets as a HuggingFace Dataset
hf_dataset = load_kaggle_disaster_csv("../data/train.csv")  # adjust path if needed

# 80/20 split with fixed seed for reproducibility
hf_dataset = hf_dataset.train_test_split(test_size=0.2, seed=42)
train_ds_raw = hf_dataset["train"]
val_ds_raw   = hf_dataset["test"]

# Create model & tokenizer
model_hf, tokenizer = create_text_classifier(num_labels=2)

# Tokenization function
def tokenize_batch(batch):
    out = tokenizer(
        batch["text"],
        padding="max_length",
        truncation=True,
        max_length=128,
    )
    out["label"] = batch["label"]
    return out

tokenized = hf_dataset.map(tokenize_batch, batched=True)
train_ds_tok = tokenized["train"]
val_ds_tok   = tokenized["test"]

# Keep only the needed columns and set PyTorch format
columns = ["input_ids", "attention_mask", "label"]
train_ds_tok.set_format(type="torch", columns=columns)
val_ds_tok.set_format(type="torch", columns=columns)


### Wrap dataset

In [ ]:
# Wrap the HF datasets so each batch is: ((input_ids, attention_mask), label)

train_wrapped = TextDatasetWrapper(train_ds_tok)
val_wrapped   = TextDatasetWrapper(val_ds_tok)

train_loader = DataLoader(train_wrapped, batch_size=16, shuffle=True)
val_loader   = DataLoader(val_wrapped,   batch_size=16, shuffle=False)

### Wrap model

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = TextClassificationWrapper(model_hf).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)

EPOCHS = 3

train_loss_hist = []
train_acc_hist  = []
val_loss_hist   = []
val_acc_hist    = []

print(f"Using device: {device}")

### Training loop

In [ ]:
for epoch in range(EPOCHS):
    print(f"\nEpoch {epoch+1}/{EPOCHS}")

    # train_one_epoch and evaluate are imported from training/utils.py.
    # They expect each batch to be (inputs, labels), where `inputs` is a tuple.
    train_loss, train_acc = train_one_epoch(
        model=model,
        loader=train_loader,
        optimizer=optimizer,
        criterion=criterion,
        device=device,
    )

    val_loss, val_acc, _, _ = evaluate(
        model=model,
        loader=val_loader,
        criterion=criterion,
        device=device,
    )

    train_loss_hist.append(train_loss)
    train_acc_hist.append(train_acc)
    val_loss_hist.append(val_loss)
    val_acc_hist.append(val_acc)

    print(
        f"train_loss={train_loss:.4f}, train_acc={train_acc:.4f}, "
        f"val_loss={val_loss:.4f}, val_acc={val_acc:.4f}"
    )

print("\nTraining complete.")


### Save model and tokenizer

In [ ]:
os.makedirs("../checkpoints/text_branch", exist_ok=True)
model_hf.save_pretrained("../checkpoints/text_branch")
tokenizer.save_pretrained("../checkpoints/text_branch")

print("Saved text model and tokenizer to ../checkpoints/text_branch")

### Plot training/eval metrics

In [ ]:
epochs = np.arange(1, EPOCHS + 1)

fig, ax = plt.subplots(1, 2, figsize=(12, 4))

# Loss curves
ax[0].plot(epochs, train_loss_hist, marker="o", label="Train")
ax[0].plot(epochs, val_loss_hist,   marker="o", label="Val")
ax[0].set_title("Loss")
ax[0].set_xlabel("Epoch")
ax[0].set_ylabel("Loss")
ax[0].legend()

# Accuracy curves
ax[1].plot(epochs, train_acc_hist, marker="o", label="Train")
ax[1].plot(epochs, val_acc_hist,   marker="o", label="Val")
ax[1].set_title("Accuracy")
ax[1].set_xlabel("Epoch")
ax[1].set_ylabel("Accuracy")
ax[1].legend()

plt.tight_layout()
plt.show()

### Evaluation on validation set

In [ ]:
# Re-run evaluate to get predictions, or cache last val_preds/val_true if you prefer
val_loss, val_acc, val_preds, val_true = evaluate(
    model=model,
    loader=val_loader,
    criterion=criterion,
    device=device,
)

val_f1 = f1_score(val_true, val_preds, average="binary")

print("\n--- DISTILBERT TEXT MODEL VALIDATION RESULTS ---")
print(f"Val Accuracy: {val_acc * 100:.2f}%")
print(f"Val F1 Score: {val_f1:.4f}")

cm = confusion_matrix(val_true, val_preds)
disp = ConfusionMatrixDisplay(cm, display_labels=["Not disaster (0)", "Disaster (1)"])

fig, ax = plt.subplots(figsize=(6, 6))
disp.plot(cmap="Blues", ax=ax, colorbar=False)
plt.title("DistilBERT Text Model - Confusion Matrix (Validation)", fontsize=14)
plt.grid(False)
plt.show()
